In [ ]:
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import classification_report
from sklearn.preprocessing import LabelEncoder
from collections import Counter

# 1. Load your dataset
df = pd.read_csv('/content/drive/MyDrive/dataset.csv')  # Change filename if needed                          best till now 80% acc 80.5% recall

# 2. Set your target column
target_col = 'Dropout_Status'  # Use your actual column name

# 3. Feature/target split
X = df.drop(target_col, axis=1)
y = df[target_col]

# 4. Encode categorical features
for col in X.select_dtypes(include=['object', 'category']):
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))

# 5. Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

# 6. Class weighting for imbalance
class_counts = Counter(y_train)
scale_pos_weight = class_counts[0] / class_counts[1]

# 7. Hyperparameter search
param_dist = {
    'num_leaves': [20, 31, 50, 70],
    'max_depth': [-1, 5, 10, 20],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'n_estimators': [100, 300, 500],
    'min_child_samples': [10, 20, 40],
    'subsample': [0.7, 0.8, 1.0]
}
lgbm = LGBMClassifier(scale_pos_weight=scale_pos_weight)
search = RandomizedSearchCV(
    lgbm, param_distributions=param_dist,
    n_iter=20, scoring='recall',  # or 'f1'
    cv=3, verbose=2, random_state=42, n_jobs=-1
)

search.fit(X_train, y_train)
from sklearn.metrics import recall_score

# 'best_model' is the result from your search.fit()
# 'X_train', 'y_train', 'X_test', 'y_test' are from your train_test_split

# 1. Make predictions on the TRAINING data
y_train_pred = lgbm.predict(X_train)
train_recall = recall_score(y_train, y_train_pred)

# 2. Make predictions on the TEST data
y_test_pred = lgbm.predict(X_test)
test_recall = recall_score(y_test, y_test_pred)

print(f"Training Recall: {train_recall * 100:.2f}%")
print(f"Test Recall:     {test_recall * 100:.2f}%")

# 8. Predict and evaluate
best_model = search.best_estimator_
y_pred = best_model.predict(X_test)
print(classification_report(y_test, y_pred))

Fitting 3 folds for each of 20 candidates, totalling 60 fits
[LightGBM] [Info] Number of positive: 917, number of negative: 3083
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000418 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1068
[LightGBM] [Info] Number of data points in the train set: 4000, number of used features: 8
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.229250 -> initscore=-1.212551
[LightGBM] [Info] Start training from score -1.212551


NotFittedError: Estimator not fitted, call fit before exploiting the model.

In [ ]:
# --- This part is correct ---
# (Imports, data loading, splitting, and search setup are all correct)
#
# 7. Run the hyperparameter search
search.fit(X_train, y_train)


# --- CORRECTED EVALUATION FLOW ---

# 8. Get the best model FROM THE SEARCH
best_model = search.best_estimator_

# 9. Now, check for overfitting using the BEST model
print("--- Checking for Overfitting on the Best Model ---")
# Make predictions on the TRAINING data using the best model
y_train_pred = best_model.predict(X_train)
train_recall = recall_score(y_train, y_train_pred)

# Make predictions on the TEST data using the best model
y_test_pred = best_model.predict(X_test)
test_recall = recall_score(y_test, y_test_pred)

print(f"Training Recall: {train_recall * 100:.2f}%")
print(f"Test Recall:     {test_recall * 100:.2f}%")


# 10. Print the final classification report for the BEST model
print("\n--- Final Classification Report ---")
# Note: y_test_pred was already calculated above, but for clarity we show it again
# y_pred = best_model.predict(X_test)
print(classification_report(y_test, y_test_pred))

In [ ]:
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import classification_report, recall_score
from sklearn.preprocessing import LabelEncoder
from collections import Counter

# 1. Load your dataset
# Make sure the path to your CSV file is correct
df = pd.read_csv('/content/drive/MyDrive/dataset.csv')

# 2. Set your target column and prepare data
target_col = 'Dropout_Status'
if 'studentID' in df.columns:
    df = df.drop('studentID', axis=1)
X = df.drop(target_col, axis=1)
y = df[target_col]

# 3. Encode categorical features
for col in X.select_dtypes(include=['object', 'category']):
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))

# 4. Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 5. Class weighting for imbalance
class_counts = Counter(y_train)
scale_pos_weight = class_counts[0] / class_counts[1]

# 6. Hyperparameter search setup
param_dist = {
    'num_leaves': [20, 31, 50, 70],
    'max_depth': [-1, 5, 10, 20],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'n_estimators': [100, 300, 500],
    'min_child_samples': [10, 20, 40],
    'subsample': [0.7, 0.8, 1.0]
}
lgbm = LGBMClassifier(scale_pos_weight=scale_pos_weight, random_state=42)
search = RandomizedSearchCV(
    lgbm, param_distributions=param_dist,
    n_iter=20, scoring='recall',
    cv=3, verbose=2, random_state=42, n_jobs=-1
)

# 7. Run the search
print("--- Starting Hyperparameter Tuning... ---")
search.fit(X_train, y_train)
print("Tuning complete.")

# 8. Get the best model
best_model = search.best_estimator_
print(f"\nBest Parameters found: {search.best_params_}")


# 9. Check for overfitting using the best model
print("\n--- Checking for Overfitting ---")
y_train_pred = best_model.predict(X_train)
train_recall = recall_score(y_train, y_train_pred)

y_test_pred = best_model.predict(X_test)
test_recall = recall_score(y_test, y_test_pred)

print(f"Training Recall: {train_recall * 100:.2f}%")
print(f"Test Recall:     {test_recall * 100:.2f}%")

# 10. Print final evaluation report
print("\n--- Final Classification Report on Test Data ---")
print(classification_report(y_test, y_test_pred))

--- Starting Hyperparameter Tuning... ---
Fitting 3 folds for each of 20 candidates, totalling 60 fits
[LightGBM] [Info] Number of positive: 912, number of negative: 3088
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000344 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 813
[LightGBM] [Info] Number of data points in the train set: 4000, number of used features: 7
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.228000 -> initscore=-1.219639
[LightGBM] [Info] Start training from score -1.219639
Tuning complete.

Best Parameters found: {'subsample': 1.0, 'num_leaves': 31, 'n_estimators': 300, 'min_child_samples': 40, 'max_depth': 20, 'learning_rate': 0.01}

--- Checking for Overfitting ---
Training Recall: 86.84%
Test Recall:     69.30%

--- Final Classification Report on Test Data ---
              precision    recall  f1-score   sup

In [ ]:
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import classification_report
from sklearn.preprocessing import LabelEncoder
from collections import Counter

# 1. Load your dataset
df = pd.read_csv('/content/drive/MyDrive/dataset.csv')  # Change filename if needed                          best till now 80% acc 80.5% recall

# 2. Set your target column
target_col = 'Dropout_Status'  # Use your actual column name

# 3. Feature/target split
X = df.drop(target_col, axis=1)
y = df[target_col]

# 4. Encode categorical features
for col in X.select_dtypes(include=['object', 'category']):
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))

# 5. Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

# 6. Class weighting for imbalance
class_counts = Counter(y_train)
scale_pos_weight = class_counts[0] / class_counts[1]

# 7. Hyperparameter search
param_dist = {
    'num_leaves': [20, 31, 50, 70],
    'max_depth': [-1, 5, 10, 20],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'n_estimators': [100, 300, 500],
    'min_child_samples': [10, 20, 40],
    'subsample': [0.7, 0.8, 1.0]
}
lgbm = LGBMClassifier(scale_pos_weight=scale_pos_weight)
search = RandomizedSearchCV(
    lgbm, param_distributions=param_dist,
    n_iter=20, scoring='recall',  # or 'f1'
    cv=3, verbose=2, random_state=42, n_jobs=-1
)
search.fit(X_train, y_train)

# 8. Predict and evaluate
best_model = search.best_estimator_
y_pred = best_model.predict(X_test)
print(classification_report(y_test, y_pred))

Fitting 3 folds for each of 20 candidates, totalling 60 fits
[LightGBM] [Info] Number of positive: 917, number of negative: 3083
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000311 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1068
[LightGBM] [Info] Number of data points in the train set: 4000, number of used features: 8
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.229250 -> initscore=-1.212551
[LightGBM] [Info] Start training from score -1.212551
              precision    recall  f1-score   support

           0       0.93      0.80      0.86       777
           1       0.53      0.80      0.64       223

    accuracy                           0.80      1000
   macro avg       0.73      0.80      0.75      1000
weighted avg       0.84      0.80      0.81      1000

